## SQL Analysis
### 1. Database Connection
### Objective

Connect Python environment with SQL database to perform business analysis using SQL queries.

In [1]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine,text

In [2]:
#Step 1: Connect to PostgreSQL

username = "postgres" # default user
password = "cjack004" # the password you set during installation
host =  "localhost" #if running Locally
port = "5432" # default PostgreSQL port
database= "global_market" #the database you created in pgAdmin

# 3. Create the database connection engine

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)
connection = engine.connect()
print("Database connected successfully")

Database connected successfully


## 2. Load Clean Data into SQL
### Objective

Store cleaned datasets into SQL tables for analysis.

## Tables Created:
### Job Listings Table
Contains job demand information:
* job_id
* title
* industry
* company_size
* company_tier
* country
* city
* remote_policy
* experience_level
* education_required
* skills
* tech_stack
* employment_type
* posted_date
* application_deadline

### Recruitment KPIs Table

Contains hiring performance metrics:

* job_id
* num_applicants
* avg_response_hours
* offer_extended
* offer_accepted

### Retention KPIs Table

Contains employee retention information:

* job_id
* starting_salary
* tenure_months
* churned_within_1yr


In [3]:
job_listings = pd.read_csv(r"C:\Users\babus\Data_Spark\Project\Global _market\Deployment\data\processed\job_listings.csv")
retention_kpis = pd.read_csv(r"C:\Users\babus\Data_Spark\Project\Global _market\Deployment\data\processed\retention_kpis.csv")
recruitment_kpis = pd.read_csv(r"C:\Users\babus\Data_Spark\Project\Global _market\Deployment\data\processed\recruiting_kpis.csv")

## Load Data:

In [4]:
# Load DataFrame into PostgreSQL

print("1. Uploading data to PostgreSQL...")

job_listings.to_sql( 
    'jobs',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10000
)

recruitment_kpis.to_sql(
    'recruitment',
    engine,
    if_exists="replace",
    index=False,
    chunksize=10000
)

retention_kpis.to_sql(
    'retention',
    engine,
    if_exists="replace",
    index=False,
    chunksize=10000
)

print("2. Setting up Primary Keys for instant query speeds...")

# Apply Primary Keys via SQL commands
with engine.begin() as connection:
    # Set job_id as primary key for 'jobs' table
    connection.execute(text("ALTER TABLE jobs ADD PRIMARY KEY (job_id);"))
    
    # Set job_id as primary key for 'recruitment' table
    connection.execute(text("ALTER TABLE recruitment ADD PRIMARY KEY (job_id);"))
    
    # Set job_id as primary key for 'retention' table
    connection.execute(text("ALTER TABLE retention ADD PRIMARY KEY (job_id);"))

print("\n Success! Data loaded and Primary Keys created successfully.")

1. Uploading data to PostgreSQL...
2. Setting up Primary Keys for instant query speeds...

 Success! Data loaded and Primary Keys created successfully.


# SQL Business Questions 


## 1. Job Listings Analysis


### Job Market Demand Analysis

1. Which industries have the highest number of job postings and what percentage of total market demand do they represent?

2. Which job titles rank in the top 10 most demanded roles by number of postings?

3. Which countries have the highest hiring activity compared to the global average?

4. Which cities have the highest job concentration within each country?

5. Which company sizes generate the most job opportunities?

6. Which industries have the highest remote job availability?

7. How does employment type distribution vary across different industries?

8. Which experience levels have the highest demand within each industry?

9. Which skills appear most frequently in job postings?

10. Which technology stacks are most commonly required for each job role?

11. Which job roles require the highest number of skills on average?

12. Which industries show growth in job postings over time based on posting dates?

13. Which companies/job categories have the highest competition based on applicant volume?


---

## 2. Recruitment KPI Analysis


### Hiring Performance Analysis


14. Which job roles receive the highest average number of applicants?

15. Which industries have the highest applicant-to-offer conversion rate?

16. What is the overall offer acceptance percentage?

17. Which job titles have the highest offer acceptance rate?

18. Which industries have above-average hiring success rates?

19. Does faster recruiter response time improve offer acceptance?

20. Which employment types have the highest recruitment conversion?

21. Which experience levels receive the most successful offers?

22. Which job roles have the longest recruitment cycle?

23. Rank industries based on recruitment efficiency.

24. Which roles have high applicant volume but low offer acceptance?

25. Which industries attract many applicants but have poor hiring conversion?


---

## 3. Retention KPI Analysis


### Employee Retention & Turnover Analysis


26. What is the overall employee churn rate?

27. Which industries have the highest employee turnover rate?

28. Which job roles have the highest churn risk?

29. Does salary level impact employee retention?

30. Compare average salary between retained and churned employees.

31. Which experience levels have the highest turnover?

32. Does company size influence employee retention?

33. Which industries have better employee stability?

34. Does employee tenure differ between retained and churned employees?

35. Which salary ranges have the highest churn percentage?

36. Rank job roles based on employee retention performance.

37. Which factors contribute most to employee turnover?

38. Which combination of industry, role, and company size shows the highest churn risk?

39. Identify employees/groups with high turnover probability based on multiple factors.


---


# Advanced SQL Techniques Used

- INNER JOIN
- LEFT JOIN
- GROUP BY
- HAVING
- CASE WHEN
- Aggregate Functions
- Common Table Expressions (CTE)
- Subqueries
- Window Functions
- RANK()
- DENSE_RANK()
- Percentage Calculations
- Date Functions

In [5]:
query = """
SELECT
    industry,
    COUNT(*) AS total_jobs
FROM jobs
GROUP BY industry
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)

result

,industry,total_jobs
0,Manufacturing,35776
1,Technology,35772
2,Education,35727
3,Transportation,35725
4,Retail,35719
5,Healthcare,35696
6,Finance,35585


In [6]:
query = """SELECT * FROM jobs """
result = pd.read_sql(query, engine)

result

,job_id,title,industry,company_size,company_tier,country,city,remote_policy,experience_level,education_required,skills,tech_stack,employment_type,posted_date,application_deadline
0,1,AI Researcher,Finance,51-200,Scale-up,Afghanistan,Jessicahaven,Remote,Senior,Master's,"Python, TensorFlow, JavaScript, Kubernetes, AW...","Node.js, Spring, Spark",Contract,2024-03-07,2024-04-16
1,2,AI Researcher,Education,501-1000,Enterprise,Afghanistan,Jenniferfurt,Remote,Senior,Not-specified,"Excel, Docker, JavaScript, SQL, TensorFlow, Java","Django, Flask",Internship,2023-10-22,2023-11-12
2,3,AI Researcher,Finance,51-200,Scale-up,Afghanistan,North Charles,Onsite,Mid,Not-specified,"Excel, Java, AWS","React, Node.js, Spring",Internship,2024-12-08,2025-01-04
3,4,AI Researcher,Technology,501-1000,Enterprise,Afghanistan,Benjaminhaven,Hybrid,Entry,Master's,"Pandas, GCP, Docker, Java, JavaScript","Node.js, Airflow, MongoDB",Full-time,2025-07-08,2025-09-02
4,5,AI Researcher,Healthcare,501-1000,Enterprise,Afghanistan,Port Daniel,Onsite,Entry,Bachelor's,"Python, Pandas, Java","PostgreSQL, Flask, MongoDB",Full-time,2024-01-04,2024-02-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,249996,UX Designer,Manufacturing,1001-5000,Corporation,Zimbabwe,Deborahville,Hybrid,Mid,Not-specified,"Java, Kubernetes, AWS","Spark, Airflow, Spring",Part-time,2025-03-22,2025-04-20
249996,249997,UX Designer,Retail,501-1000,Enterprise,Zimbabwe,Jonesberg,Remote,Senior,Not-specified,"Python, TensorFlow, Kubernetes, Excel, Docker,...","Spark, Node.js",Full-time,2024-01-11,2024-02-06
249997,249998,UX Designer,Manufacturing,501-1000,Enterprise,Zimbabwe,Myersshire,Hybrid,Senior,PhD,"AWS, Python, GCP","Spark, Spring",Full-time,2024-10-09,2024-10-17
249998,249999,UX Designer,Healthcare,51-200,Scale-up,Zimbabwe,South Phillipside,Hybrid,Mid,Bachelor's,"Python, SQL, TensorFlow","MongoDB, Django",Part-time,2024-02-07,2024-03-04
